# Customer Segmentation & Business Intelligence (SQL)

After cleaning the dataset and engineering customer-level metrics in Python, the next step is to use SQL to answer the brand's core business questions.

The engineered metrics provide a structured way to evaluate customer value, loyalty, promotional dependence, and revenue quality.

In [1]:
import pandas as pd
import psycopg2
import getpass

conn = psycopg2.connect(
    host="localhost",
    database="summer_project_db",
    user="postgres",
    password=getpass.getpass("Enter PostgreSQL password: ")
)

## Query 1: Loyalty vs Discount Dependency

### Business Question
Who are genuinely loyal customers and who only purchase when discounts are available?

### Why This Matters
The central challenge of the project is determining whether the business is creating true customer loyalty or simply generating sales through promotions.

Using:
- BLS to measure loyalty behavior
- PDI to measure discount reliance

Customers are classified into four segments:
- Genuine Loyal
- Discount Loyal
- Bargain Hunter
- Occasional Customer

This analysis helps determine which customer groups can be retained without aggressive discounting.

In [12]:
query = """
SELECT
    CASE
        WHEN BLS >= 0.70 AND PDI <= 0.30
            THEN 'Genuine Loyal'
        WHEN BLS >= 0.70 AND PDI > 0.30
            THEN 'Discount Loyal'
        WHEN BLS < 0.70 AND PDI > 0.50
            THEN 'Bargain Hunter'
        ELSE 'Occasional Customer'
    END AS customer_segment,

    COUNT(*) AS customers,
    ROUND(AVG(purchase_amount),2) AS avg_spend,
    ROUND(AVG(previous_purchases),2) AS avg_previous_purchases,
    ROUND(AVG(e_LTV),2) AS avg_ltv

FROM customer_data
GROUP BY customer_segment
ORDER BY avg_ltv DESC;
"""

pd.read_sql(query, conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\542373858.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,customer_segment,customers,avg_spend,avg_previous_purchases,avg_ltv
0,Genuine Loyal,462,60.22,43.00,2659.84
1,Discount Loyal,378,59.48,43.16,2633.24
2,Occasional Customer,1761,60.11,20.35,1285.62
3,Bargain Hunter,1299,59.22,20.68,1281.46


## Query 2: Behavioral Drivers of Customer Value

### Business Question
What behavioral patterns are associated with high-value customers?

### Why This Matters
The dataset does not contain timestamps or churn labels. Therefore, customer value must be inferred using behavioral indicators.

This query compares customer value tiers against:
- Behavioral Loyalty Score (BLS)
- Economic Loyalty Score (ELS)
- Revenue Quality Index (RQI)
- Previous Purchases
- Review Ratings

The objective is to identify which behaviors consistently appear among the highest-value customers.

In [13]:
query="""
SELECT
    customer_value_tier,

    ROUND(AVG(BLS),3) AS avg_bls,
    ROUND(AVG(ELS),3) AS avg_els,
    ROUND(AVG(RQI),3) AS avg_rqi,
    ROUND(AVG(previous_purchases),2) AS avg_prev_purchases,
    ROUND(AVG(review_rating),2) AS avg_rating

FROM customer_data
GROUP BY customer_value_tier
ORDER BY avg_bls DESC;
"""
pd.read_sql(query,conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\366038913.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,customer_value_tier,avg_bls,avg_els,avg_rqi,avg_prev_purchases,avg_rating
0,Platinum,0.758,0.622,0.746,43.90,3.81
1,Gold,0.645,0.527,0.656,35.65,3.78
2,Silver,0.519,0.414,0.534,27.06,3.72
3,Bronze,0.278,0.367,0.498,10.03,3.75


## Query 3: Retention Categories

### Business Question
Which product categories are most associated with repeat purchasing behavior?

### Why This Matters
Some product categories naturally encourage repeat purchases and customer retention.

By examining:
- Previous Purchases
- BLS
- eLTV

we can identify categories that appear most frequently among loyal, long-term customers.

These categories may deserve increased investment, inventory focus, and retention marketing.

In [14]:
query= """
SELECT
    category,

    ROUND(AVG(previous_purchases),2) AS avg_prev_purchases,
    ROUND(AVG(BLS),2) AS avg_bls,
    ROUND(AVG(e_LTV),2) AS avg_ltv,
    COUNT(*) AS customers

FROM customer_data
GROUP BY category
ORDER BY avg_prev_purchases DESC;
"""

pd.read_sql(query,conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\1913759090.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,category,avg_prev_purchases,avg_bls,avg_ltv,customers
0,Accessories,25.73,0.50,1589.02,1240
1,Footwear,25.23,0.50,1588.04,599
2,Clothing,25.20,0.49,1583.52,1737
3,Outerwear,24.96,0.49,1483.36,324


## Query 4: Acquisition Categories

### Business Question
Which product categories appear most often among lower-tenure customers?

### Why This Matters
Not every category serves the same purpose.

Some categories act as:
- Customer acquisition products

while others act as:
- Customer retention products

Categories with lower average purchase history may represent customer entry points into the brand.

In [15]:
query = """
SELECT
    category,

    ROUND(AVG(previous_purchases),2) AS avg_prev_purchases,
    COUNT(*) AS customers

FROM customer_data
GROUP BY category
ORDER BY avg_prev_purchases ASC;
"""

pd.read_sql(query,conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\1760067871.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,category,avg_prev_purchases,customers
0,Outerwear,24.96,324
1,Clothing,25.20,1737
2,Footwear,25.23,599
3,Accessories,25.73,1240


## Query 5: Seasonal Customer Analysis

### Business Question
Which seasons are associated with stronger customer retention?

### Why This Matters
The project requires understanding whether purchasing behavior varies across seasons.

This query evaluates:
- Previous Purchases
- Loyalty Scores
- Estimated Lifetime Value

across each season.

The findings can guide seasonal marketing and inventory planning decisions.

In [16]:
query="""
SELECT
    season,

    ROUND(AVG(previous_purchases),2) AS avg_prev_purchases,
    ROUND(AVG(BLS),2) AS avg_bls,
    ROUND(AVG(e_LTV),2) AS avg_ltv

FROM customer_data
GROUP BY season
ORDER BY avg_prev_purchases DESC;
"""

pd.read_sql(query,conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\3676174514.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,season,avg_prev_purchases,avg_bls,avg_ltv
0,Winter,25.92,0.51,1623.15
1,Summer,25.41,0.49,1554.80
2,Spring,25.12,0.50,1542.08
3,Fall,24.97,0.49,1591.13


## Query 6: Geographic Opportunity Analysis

### Business Question
Which locations demonstrate genuine customer demand rather than discount-driven demand?

### Why This Matters
A high-sales region is not necessarily a strong market.

Some locations may generate revenue primarily through promotions.

This analysis compares:
- Customer Count
- Average Spend
- Promo Dependency
- Loyalty
- Estimated Lifetime Value

to identify regions where customers purchase because of brand preference rather than discounts.

In [17]:
query="""
SELECT
    location,

    COUNT(*) AS customers,

    ROUND(AVG(purchase_amount),2) AS avg_spend,
    ROUND(AVG(PDI),2) AS avg_pdi,
    ROUND(AVG(BLS),2) AS avg_bls,
    ROUND(AVG(e_LTV),2) AS avg_ltv

FROM customer_data
GROUP BY location
HAVING COUNT(*) >= 20
ORDER BY avg_ltv DESC;
"""

pd.read_sql(query,conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\3053911159.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,location,customers,avg_spend,avg_pdi,avg_bls,avg_ltv
0,Arizona,65,66.55,0.34,0.54,2000.77
1,Pennsylvania,74,66.57,0.45,0.53,1937.86
2,Alaska,72,67.60,0.40,0.55,1937.71
3,Hawaii,65,57.72,0.49,0.57,1780.91
4,Nevada,87,63.38,0.47,0.50,1780.66
5,Wyoming,71,60.69,0.42,0.54,1764.69
6,Iowa,69,60.88,0.52,0.53,1755.70
7,Michigan,73,62.10,0.40,0.50,1753.89
8,Utah,71,62.58,0.46,0.51,1742.99
9,Tennessee,77,61.97,0.36,0.52,1717.19


## Query 7: Demographic Opportunity Analysis

### Business Question
Which demographic groups are commercially underleveraged?

### Why This Matters
High-value customer segments are not always the largest segments.

By combining:
- Age Groups
- Gender
- eLTV

we can identify demographic groups with strong customer value but relatively low representation.

These segments may present future growth opportunities.

In [18]:
query="""
SELECT
    gender,

    CASE
        WHEN age < 25 THEN '18-24'
        WHEN age < 35 THEN '25-34'
        WHEN age < 45 THEN '35-44'
        ELSE '45+'
    END AS age_group,

    COUNT(*) AS customers,
    ROUND(AVG(e_LTV),2) AS avg_ltv

FROM customer_data
GROUP BY gender, age_group
ORDER BY avg_ltv DESC;
"""

pd.read_sql(query,conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\4135567443.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,gender,age_group,customers,avg_ltv
0,Female,25-34,241,1658.05
1,Male,45+,1307,1628.60
2,Male,35-44,497,1596.07
3,Male,18-24,334,1539.90
4,Female,35-44,232,1539.30
5,Male,25-34,514,1534.78
6,Female,45+,623,1518.82
7,Female,18-24,152,1479.16


## Query 8: Ideal Customer Profile (ICP)

### Business Question
What does the brand's most valuable customer look like?

### Why This Matters
One of the final deliverables is a data-backed Ideal Customer Profile.

This query focuses on Platinum customers and examines:

- Gender
- Product Category
- Payment Preferences
- Subscription Status
- Loyalty
- Customer Value

The output helps define the customer type the business should actively acquire and retain.

In [19]:
query="""
SELECT
    gender,
    category,
    payment_method,
    subscription_status,

    COUNT(*) AS customers,

    ROUND(AVG(e_LTV),2) AS avg_ltv,
    ROUND(AVG(BLS),2) AS avg_bls,
    ROUND(AVG(review_rating),2) AS avg_rating

FROM customer_data
WHERE customer_value_tier = 'Platinum'

GROUP BY
    gender,
    category,
    payment_method,
    subscription_status

ORDER BY avg_ltv DESC
LIMIT 10;
"""

pd.read_sql(query,conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\3759590980.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,gender,category,payment_method,subscription_status,customers,avg_ltv,avg_bls,avg_rating
0,Female,Outerwear,PayPal,False,1,4950.00,0.84,2.60
1,Female,Footwear,Debit Card,False,2,4464.00,0.67,3.75
2,Male,Footwear,Bank Transfer,True,1,4410.00,0.90,3.80
3,Male,Footwear,Bank Transfer,False,3,4372.67,0.79,3.50
4,Male,Outerwear,Credit Card,True,1,4335.00,0.93,4.60
5,Male,Footwear,Credit Card,False,5,4262.80,0.79,4.22
6,Female,Outerwear,Debit Card,False,2,4256.00,0.74,4.25
7,Male,Clothing,Debit Card,True,9,4250.67,0.79,3.71
8,Male,Outerwear,Bank Transfer,True,1,4224.00,0.68,2.60
9,Male,Clothing,PayPal,False,12,4185.50,0.75,4.03


## Query 9: Promotional Sunset Candidates

### Business Question
Which customer segments can tolerate reduced discounting?

### Why This Matters
The project requires recommendations for reducing discount dependency without damaging sales volume.

By comparing:
- Promo Dependency Index (PDI)
- Loyalty Scores (BLS)
- Customer Value (eLTV)

across value tiers, we can identify customer groups that are likely to continue purchasing even if promotional activity is reduced.

These findings form the foundation of the Promotional Sunset Plan.

In [20]:
query="""
SELECT
    customer_value_tier,

    ROUND(AVG(PDI),2) AS avg_pdi,
    ROUND(AVG(BLS),2) AS avg_bls,
    ROUND(AVG(e_LTV),2) AS avg_ltv

FROM customer_data
GROUP BY customer_value_tier
ORDER BY avg_ltv DESC;
"""

pd.read_sql(query,conn)

C:\Users\being\AppData\Local\Temp\ipykernel_9444\2075976555.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query,conn)


,customer_value_tier,avg_pdi,avg_bls,avg_ltv
0,Platinum,0.44,0.76,3927.39
1,Gold,0.42,0.64,2584.50
2,Silver,0.44,0.52,1349.54
3,Bronze,0.41,0.28,428.34
